# Part II · Lesson 5 — Compare travel options with a tool loop

### Import the graph helpers

In [ ]:
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END

### Import the model and messages

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

### Connect to our existing Groq model

In [ ]:
# Keep GROQ_API_KEY in the local .env file.
load_dotenv()
model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
    reasoning_effort="low",
)

### Import the tool helpers

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition

### Describe the travel request

In [ ]:
class TravelState(TypedDict):
    origin: str
    destination: str
    budget: int
    messages: Annotated[list, add_messages]
    status: str

### Find the available route IDs

In [ ]:
@tool
def find_routes(origin: str, destination: str) -> list[str]:
    """Find sample route IDs between an origin and destination."""
    if origin.lower() == "mumbai" and destination.lower() == "goa":
        return ["TRAIN-01", "FLIGHT-01"]
    return []

### Read a route's details

In [ ]:
@tool
def get_route_details(route_id: str) -> dict:
    """Read the sample price, duration, and availability for a returned route ID."""
    routes = {
        "TRAIN-01": {"mode": "train", "price": 900, "duration": "10 hours", "available": True},
        "FLIGHT-01": {"mode": "flight", "price": 4500, "duration": "1.5 hours", "available": True},
    }
    return routes[route_id]

### Bind both tools

In [ ]:
travel_tools = [find_routes, get_route_details]
tool_model = model.bind_tools(travel_tools)

### Let the model decide what it needs next

In [ ]:
def assistant(state: TravelState) -> dict:
    reply = tool_model.invoke([
        SystemMessage(
            f"Compare {state['origin']} to {state['destination']} travel "
            f"within a budget of INR {state['budget']}. Find routes first, "
            "then get details for the returned IDs before comparing. Use the tools."
        ),
        *state["messages"],
    ])
    status = "working" if reply.tool_calls else "answered"
    return {"messages": [reply], "status": status}

### Connect the model-and-tool loop

In [ ]:
builder = StateGraph(TravelState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(travel_tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant", tools_condition, {"tools": "tools", END: END}
)
builder.add_edge("tools", "assistant")
graph = builder.compile()

### Run a comparison with an execution limit

In [ ]:
travel_input = {
    "origin": "Mumbai", "destination": "Goa", "budget": 2000,
    "messages": [HumanMessage("Compare the available trains and flights within my budget.")],
    "status": "new",
}
response = graph.invoke(travel_input, {"recursion_limit": 12})
print(response["messages"][-1].text)

### Read the visible execution trace

In [ ]:
for message in response["messages"]:
    print(message.type, message.text)
    if getattr(message, "tool_calls", None):
        print("Requested calls:", message.tool_calls)